# 🧩 Mini-Lab: Hosted Backend (Fixture Path Required)

**Module 6: The Replaceable Model Interface** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Call** a hosted AI API using a base URL, an environment-variable key, and an explicit timeout
2. **Apply** secrets hygiene by loading API keys from environment variables instead of hard-coding them
3. **Implement** a hosted backend that conforms to `AIClient` and **runs against recorded fixtures when no key is present**
4. **Parse** API responses defensively

## Target Concepts

| Concept | Gloss |
|---------|-------|
| HTTP API Request | Base URL, env-var key, explicit timeout, status handling |
| Secrets Hygiene | Keep keys out of code, logs, and git |
| Interface Seam | Adapter returns your dataclass, never the vendor object |
| Output Parsing | Model output may not be valid JSON; parse and guard |


## Recap

In **mini-define-ai-protocol**, we defined an `AIClient` protocol with `complete(AIRequest) -> AIResponse` and built mock/echo backends. In **mini-typed-prompt-parser**, we learned to parse LLM responses defensively.

Now we'll build a **real hosted backend** — one that actually calls an API over HTTP. This is "Backend 2" in our module: a class that speaks HTTP to a remote service but still satisfies the same `AIClient` protocol. The business logic won't know the difference.

## Step 1 — Shared Types from Previous Labs

We reuse the exact `AIRequest`, `AIResponse`, and `AIClient` protocol from **mini-define-ai-protocol**.

In [1]:
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable, Optional


@dataclass
class AIRequest:
    """What the business logic sends to the AI backend."""
    prompt: str
    system: str = "You are a helpful assistant."
    temperature: float = 0.0
    max_tokens: int = 256


@dataclass
class AIResponse:
    """What the AI backend returns to the business logic."""
    text: str
    model: str = "unknown"
    usage: dict = field(default_factory=dict)


@runtime_checkable
class AIClient(Protocol):
    """Any AI backend must implement this single method."""
    def complete(self, request: AIRequest) -> AIResponse: ...


print("Shared types ready: AIRequest, AIResponse, AIClient protocol.")

Shared types ready: AIRequest, AIResponse, AIClient protocol.


## Step 2 — Secrets Hygiene: Loading the API Key

The first rule of secrets hygiene: **never hard-code an API key**. Load it from an environment variable, and fail loudly if it's missing. Never print or log the key itself.

In [2]:
import os
from dotenv import load_dotenv

# Load .env file (if it exists) into environment variables
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    # Show only the first 5 and last 3 characters — never the full key
    masked = api_key[:5] + "..." + api_key[-3:]
    print(f"✅ API key loaded (masked): {masked}")
else:
    print("⚠️  OPENAI_API_KEY not found in environment.")
    print("   Set it in a .env file or export it in your shell.")

✅ API key loaded (masked): sk-pr...KoA


**Why masking matters:** Even in notebooks, screenshots and screen recordings can leak keys. Always mask before printing. In production logging, never include the key at all.

## Step 3 — The Hosted Backend: Anatomy of an HTTP API Call

A hosted AI API call has four ingredients:

1. **Base URL** — the endpoint (`https://api.openai.com/v1`)
2. **Auth header** — `Authorization: Bearer <key>`
3. **Request body** — JSON payload with model, messages, parameters
4. **Timeout** — how long to wait before giving up

Let's build a backend class that wraps all four behind the `AIClient` protocol.

In [3]:
import requests


class HostedAIClient:
    """AI backend that calls a hosted API over HTTP."""

    def __init__(
        self,
        api_key: str,
        base_url: str = "https://api.openai.com/v1",
        model: str = "gpt-4o-mini",
        timeout: int = 30,
    ):
        if not api_key:
            raise ValueError("API key must not be empty.")
        self.api_key = api_key
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.timeout = timeout

    def complete(self, request: AIRequest) -> AIResponse:
        """Send a request to the hosted API and return an AIResponse."""
        url = f"{self.base_url}/chat/completions"

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": request.system},
                {"role": "user", "content": request.prompt},
            ],
            "temperature": request.temperature,
            "max_tokens": request.max_tokens,
        }

        # The actual HTTP call — with an explicit timeout
        resp = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
        resp.raise_for_status()  # raises on 4xx/5xx

        # Defensive parsing of the response JSON
        body = resp.json()
        text = body.get("choices", [{}])[0].get("message", {}).get("content", "")
        usage = body.get("usage", {})
        model_used = body.get("model", self.model)

        return AIResponse(text=text.strip(), model=model_used, usage=usage)


print("HostedAIClient defined.")
print("Key ingredients: base_url, api_key (in auth header), payload, timeout.")

HostedAIClient defined.
Key ingredients: base_url, api_key (in auth header), payload, timeout.


### Key Design Decisions

| Decision | Why |
|----------|-----|
| `timeout=30` as explicit parameter | Prevents hanging forever if the API is slow; makes the limit visible and configurable |
| `resp.raise_for_status()` | Converts HTTP errors (401 Unauthorized, 429 Rate Limit) into Python exceptions immediately |
| `.get()` chains for response parsing | Defensive — if the response shape changes, we get empty strings instead of a `KeyError` crash |
| `api_key` passed as constructor arg | Secrets hygiene — the caller loads the key from the environment, the class never touches `os.getenv` directly |

## Step 4 — Verify Protocol Conformance

Before making a real API call, let's confirm the class satisfies the `AIClient` protocol.

In [4]:
# Use a dummy key for the isinstance check — no network call here
test_client = HostedAIClient(api_key="sk-test-dummy-key")

print(f"HostedAIClient satisfies AIClient? {isinstance(test_client, AIClient)}")
print(f"Default base_url: {test_client.base_url}")
print(f"Default model:    {test_client.model}")
print(f"Default timeout:  {test_client.timeout}s")

HostedAIClient satisfies AIClient? True
Default base_url: https://api.openai.com/v1
Default model:    gpt-4o-mini
Default timeout:  30s


## Step 5 — Recorded fixtures first, live key optional

The **required path does not need a paid API key**. `HostedAIClient` speaks HTTP when `OPENAI_API_KEY` is set. When it is missing, it returns a **recorded fixture** with the same `AIResponse` shape so the rest of the pipeline still runs.

A live call is optional homework if you have a key and a spending cap. Do not block the lab on billing.


In [5]:
# Recorded fixture — same shape as a hosted response, no network.
FIXTURES = {
    "charged twice": AIResponse(text="billing", model="gpt-4o-mini-fixture", usage={"prompt_tokens": 24, "completion_tokens": 1}),
    "default": AIResponse(text="other", model="gpt-4o-mini-fixture", usage={"prompt_tokens": 20, "completion_tokens": 1}),
}


def complete_with_fallback(client: HostedAIClient, request: AIRequest) -> AIResponse:
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key:
        needle = request.prompt.lower()
        for fragment, fixture in FIXTURES.items():
            if fragment != "default" and fragment in needle:
                print("No OPENAI_API_KEY — using recorded fixture.")
                return fixture
        print("No OPENAI_API_KEY — using default recorded fixture.")
        return FIXTURES["default"]
    return client.complete(request)


hosted = HostedAIClient(
    api_key=os.getenv("OPENAI_API_KEY", "sk-fixture"),
    model="gpt-4o-mini",
    timeout=15,
)

request = AIRequest(
    prompt="Classify this support ticket: 'I was charged twice this month'. Category:",
    system="You are a support classifier. Reply with exactly one word: billing, technical, account, or other.",
    temperature=0.0,
    max_tokens=10,
)

response = complete_with_fallback(hosted, request)
print(f"Hosted-or-fixture call succeeded")
print(f"   Category: {response.text}")
print(f"   Model:    {response.model}")


✅ Hosted API call succeeded!
   Category: billing
   Model:    gpt-4o-mini-2024-07-18


## Step 6 — What Happens When Things Go Wrong

In production, HTTP calls fail — bad keys, network timeouts, rate limits. Let's see how our backend behaves with a bad key and with a short timeout.

In [6]:
# Scenario 1: Invalid API key
bad_client = HostedAIClient(api_key="sk-invalid-key-12345", timeout=10)

try:
    bad_client.complete(request)
except requests.exceptions.HTTPError as e:
    print(f"❌ Auth error (expected): HTTP {e.response.status_code}")
    print(f"   The API rejected the invalid key.")

❌ Auth error (expected): HTTP 401
   The API rejected the invalid key.


In [7]:
# Scenario 2: Unreachable URL with a very short timeout
slow_client = HostedAIClient(
    api_key="sk-fake",
    base_url="https://httpbin.org/delay/10",  # will take 10 seconds
    timeout=2,  # but we only wait 2
)

try:
    slow_client.complete(request)
except requests.exceptions.Timeout:
    print("❌ Timeout (expected): the request exceeded the 2-second limit.")
    print("   Explicit timeouts prevent your app from hanging forever.")
except Exception as e:
    print(f"❌ Error (expected): {type(e).__name__}: {e}")

❌ Error (expected): HTTPError: 404 Client Error: NOT FOUND for url: https://httpbin.org/delay/10/chat/completions


**Key lesson:** Without an explicit `timeout`, a `requests.post()` call can hang indefinitely. Always set a timeout — it's the difference between a recoverable failure and a stuck application.

## Step 7 — The Interface Seam in Action

Let's prove the seam works: the same `classify_ticket` function from **mini-define-ai-protocol** runs identically with the hosted backend.

In [8]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


def classify_ticket(ticket_text: str, ai_client: AIClient) -> str:
    """Classify a support ticket. The AI backend is injected."""
    categories = ["billing", "technical", "account", "other"]
    request = AIRequest(
        prompt=(
            f"Classify this support ticket into one of {categories}.\n"
            f"Ticket: {ticket_text}\nCategory:"
        ),
        system="Reply with exactly one category name, nothing else.",
        temperature=0.0,
        max_tokens=10,
    )
    response = ai_client.complete(request)
    category = response.text.strip().lower()
    if category not in categories:
        category = "other"
    return category


# Run with the hosted backend
ticket = "My login keeps failing with error 403"
result = classify_ticket(ticket, hosted)

md(f"""
| Detail | Value |
|--------|-------|
| **Ticket** | {ticket} |
| **Backend** | `HostedAIClient` → `{hosted.model}` |
| **Timeout** | {hosted.timeout}s |
| **Result** | `{result}` |

The function `classify_ticket` has **zero vendor imports** — it works with
mock, echo, or hosted backends through the same `AIClient` protocol.
""")


| Detail | Value |
|--------|-------|
| **Ticket** | My login keeps failing with error 403 |
| **Backend** | `HostedAIClient` → `gpt-4o-mini` |
| **Timeout** | 15s |
| **Result** | `technical` |

The function `classify_ticket` has **zero vendor imports** — it works with
mock, echo, or hosted backends through the same `AIClient` protocol.


## 🎯 Summary

### Key Takeaways

1. **HTTP API call anatomy** — a hosted AI call needs four things: a base URL, an auth header with the API key, a JSON payload, and an explicit timeout
2. **Secrets hygiene** — load API keys from environment variables (`os.getenv`), never hard-code them, and mask before printing; the backend class receives the key as a parameter, never reads the environment itself
3. **Explicit timeout** — without a `timeout` parameter, HTTP requests can hang indefinitely; always set one to keep failures recoverable
4. **Interface seam preserved** — `HostedAIClient` satisfies the same `AIClient` protocol as mock and echo backends, so business logic stays vendor-free and testable